# OSS-Instruct Pipeline for Python Code Snippets

This notebook builds an **OSS-Instruct-style dataset** by:

1. Loading an open-source dataset from Hugging Face
2. Filtering to Python code snippets only
3. Cleaning and deduplicating snippets
4. Generating **one instruction per code snippet** using the OpenAI Batch API
5. Merging instructions back with code
6. Producing a final dataset with only:

```json
{"instruction": "...", "response": "..."}
```

The notebook is designed so that the **target final dataset size is adjustable** through `TARGET_N`.

Your thesis target can remain **20,000** examples by default.

In [ ]:
# =========================
# 1. Install dependencies
# =========================

!pip -q install datasets openai pandas tqdm tiktoken huggingface_hub

In [ ]:
# =========================
# 2. Imports
# =========================

import os
import re
import json
import ast
import hashlib
from pathlib import Path
from typing import Optional, List, Dict, Any

import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset
from openai import OpenAI

In [ ]:
# =========================
# 3. Config
# =========================

# Main target for the final cleaned dataset
TARGET_N = 20_000

# Oversample before filtering because some examples will be rejected later
RAW_MULTIPLIER = 3

# Hugging Face dataset choice
# Replace this with your preferred source dataset if needed.
# Current default is a broad permissive code dataset often used for code studies.
HF_DATASET_NAME = "codeparrot/github-code-clean"
HF_DATASET_CONFIG = "all-all"
HF_SPLIT = "train"

# Output paths
OUT_DIR = Path("oss_instruct_out")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_SNIPPETS_PATH = OUT_DIR / "01_raw_python_snippets.jsonl"
FILTERED_SNIPPETS_PATH = OUT_DIR / "02_filtered_python_snippets.jsonl"
BATCH_INPUT_PATH = OUT_DIR / "03_openai_batch_input.jsonl"
BATCH_OUTPUT_RAW_PATH = OUT_DIR / "04_batch_output_raw.jsonl"
BATCH_PARSED_PATH = OUT_DIR / "05_batch_parsed.jsonl"
FINAL_DATASET_PATH = OUT_DIR / "06_oss_instruct_final.jsonl"
REJECTS_PATH = OUT_DIR / "rejects.jsonl"

# OpenAI generation config
OPENAI_MODEL = "gpt-4.1-mini"
OPENAI_TEMPERATURE = 0.2
OPENAI_MAX_OUTPUT_TOKENS = 120

# Extraction / filtering rules
MIN_CODE_CHARS = 80
MAX_CODE_CHARS = 4000
MIN_LINES = 3
MAX_LINES = 120

# If your source dataset supports streaming, this lets you scan large datasets efficiently
USE_STREAMING = True

# For raw candidate collection
RAW_TARGET = TARGET_N * RAW_MULTIPLIER

# Field candidates commonly used for code in HF datasets.
CODE_FIELD_CANDIDATES = ["code", "content", "text", "body"]

# Python language field candidates commonly used in datasets
LANG_FIELD_CANDIDATES = ["language", "lang", "repo_language", "path"]

In [ ]:
# =========================
# 4. API setup
# =========================

# Set your key in Colab before running:
# import os
# os.environ["OPENAI_API_KEY"] = "sk-..."

client = OpenAI()

print("OpenAI client ready.")

## Notes on the default source dataset

The default notebook uses:

- `codeparrot/github-code-clean`

That is a broad open-source code dataset, but you may want to switch to another dataset if:

- you need clearer licensing metadata,
- you want stricter Python-only sourcing,
- or you want smaller, more self-contained functions.

The notebook is written so the pipeline logic stays the same even if you swap datasets.

In [ ]:
# =========================
# 5. Helper functions
# =========================

def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def normalize_whitespace(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace("\t", "    ")
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def get_first_present(record: Dict[str, Any], candidates: List[str]) -> Optional[Any]:
    for c in candidates:
        if c in record and record[c] is not None:
            return record[c]
    return None

def looks_like_python(record: Dict[str, Any], code: str) -> bool:
    language_value = get_first_present(record, LANG_FIELD_CANDIDATES)

    if isinstance(language_value, str):
        lv = language_value.lower()
        if lv == "python":
            return True
        if lv.endswith(".py"):
            return True
        if "/python/" in lv or "python" in lv:
            return True

    # Heuristic fallback based on syntax
    python_signals = [
        r"^def\s+\w+\(",
        r"^class\s+\w+\b",
        r"if\s+__name__\s*==\s*['\"]__main__['\"]",
        r"import\s+\w+",
        r"from\s+\w+\s+import\s+",
        r"elif\s+",
        r"except\s+",
        r"\bself\b",
        r"\bNone\b",
        r"\bTrue\b|\bFalse\b",
    ]
    score = 0
    for pat in python_signals:
        if re.search(pat, code, flags=re.MULTILINE):
            score += 1
    return score >= 2

def parseable_python(code: str) -> bool:
    try:
        ast.parse(code)
        return True
    except SyntaxError:
        return False
    except Exception:
        return False

def trivial_or_low_value(code: str) -> bool:
    stripped = code.strip()

    if len(stripped) < MIN_CODE_CHARS:
        return True

    line_count = len(stripped.splitlines())
    if line_count < MIN_LINES or line_count > MAX_LINES:
        return True

    if len(stripped) > MAX_CODE_CHARS:
        return True

    # Reject import-only or comment-only blocks
    nonempty = [ln.strip() for ln in stripped.splitlines() if ln.strip()]
    if not nonempty:
        return True

    if all(ln.startswith("import ") or ln.startswith("from ") for ln in nonempty):
        return True

    if all(ln.startswith("#") for ln in nonempty):
        return True

    # Reject likely tests
    if re.search(r"(^|\n)\s*def\s+test_", stripped):
        return True
    if "pytest" in stripped:
        return True
    if "unittest" in stripped:
        return True

    # Reject notebooks / shell escapes
    if re.search(r"(^|\n)!\w+", stripped):
        return True

    return False

def self_contained_enough(code: str) -> bool:
    # Light heuristic, not perfect
    bad_patterns = [
        r"pass\s*$",
        r"TODO",
        r"FIXME",
        r"your code here",
    ]
    for pat in bad_patterns:
        if re.search(pat, code, flags=re.IGNORECASE | re.MULTILINE):
            return False
    return True

def build_record(record: Dict[str, Any], code: str, idx: int) -> Dict[str, Any]:
    return {
        "id": f"oss_{idx:07d}",
        "code": code,
        "code_hash": sha256_text(code),
        "source_dataset": HF_DATASET_NAME,
        "metadata": {
            "path": record.get("path"),
            "repo_name": record.get("repo_name"),
            "license": record.get("license"),
            "language": record.get("language") or record.get("lang"),
        },
    }

def write_jsonl(path: Path, rows: List[Dict[str, Any]]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

def append_jsonl(path: Path, rows: List[Dict[str, Any]]) -> None:
    with open(path, "a", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

In [ ]:
# =========================
# 6. Load source dataset
# =========================

dataset = load_dataset(
    HF_DATASET_NAME,
    HF_DATASET_CONFIG,
    split=HF_SPLIT,
    streaming=USE_STREAMING,
)

print("Dataset loaded.")
print("Streaming:", USE_STREAMING)

In [ ]:
# =========================
# 7. Extract raw Python snippets
# =========================

accepted = []
rejects = []

seen_hashes = set()
counter = 0

iterator = dataset if USE_STREAMING else iter(dataset)

for i, record in enumerate(tqdm(iterator, total=None if USE_STREAMING else len(dataset))):
    raw_code = get_first_present(record, CODE_FIELD_CANDIDATES)
    if not isinstance(raw_code, str):
        continue

    code = normalize_whitespace(raw_code)

    if not looks_like_python(record, code):
        continue

    if trivial_or_low_value(code):
        rejects.append({"stage": "raw_filter", "reason": "trivial_or_low_value"})
        continue

    if not self_contained_enough(code):
        rejects.append({"stage": "raw_filter", "reason": "not_self_contained_enough"})
        continue

    if not parseable_python(code):
        rejects.append({"stage": "raw_filter", "reason": "not_parseable_python"})
        continue

    code_hash = sha256_text(code)
    if code_hash in seen_hashes:
        rejects.append({"stage": "raw_filter", "reason": "duplicate_code"})
        continue

    seen_hashes.add(code_hash)
    counter += 1
    accepted.append(build_record(record, code, counter))

    if len(accepted) >= RAW_TARGET:
        break

write_jsonl(RAW_SNIPPETS_PATH, accepted)
append_jsonl(REJECTS_PATH, rejects)

print(f"Collected raw Python snippets: {len(accepted):,}")
print(f"Saved to: {RAW_SNIPPETS_PATH}")

In [ ]:
# =========================
# 8. Optional second-pass filtering
# =========================

raw_rows = read_jsonl(RAW_SNIPPETS_PATH)

filtered = []
more_rejects = []

for row in tqdm(raw_rows):
    code = row["code"]

    # Keep snippets with at least one function/class or meaningful logic
    has_structure = bool(
        re.search(r"(^|\n)\s*def\s+\w+\(", code) or
        re.search(r"(^|\n)\s*class\s+\w+\b", code) or
        re.search(r"\bfor\b|\bwhile\b|\breturn\b|\bif\b", code)
    )

    if not has_structure:
        more_rejects.append({"id": row["id"], "stage": "second_pass", "reason": "no_meaningful_structure"})
        continue

    filtered.append(row)

write_jsonl(FILTERED_SNIPPETS_PATH, filtered)
append_jsonl(REJECTS_PATH, more_rejects)

print(f"Second-pass filtered rows: {len(filtered):,}")
print(f"Saved to: {FILTERED_SNIPPETS_PATH}")

## Instruction-generation prompt design

This prompt is designed to generate **one realistic user instruction** from a code snippet.

The instruction should:

- describe the task solved by the code,
- be concise,
- not mention “the code above,”
- not contain a solution,
- not contain markdown,
- and be usable directly as a training prompt.

In [ ]:
# =========================
# 9. Prompt template
# =========================

SYSTEM_PROMPT = (
    "You generate concise, realistic programming instructions from Python code snippets. "
    "Your job is to infer what instruction a user could have given that would reasonably lead to the code."
)

def make_user_prompt(code: str) -> str:
    return f"""Generate exactly one realistic user instruction for the following Python code snippet.

Requirements:
- Return only the instruction text
- Do not mention 'the code', 'the snippet', or 'above'
- Do not explain the code
- Do not include markdown
- Do not include a solution
- Make the instruction specific enough that the provided code is a reasonable answer
- Keep it concise and natural

Python code:
{code}
"""

In [ ]:
# =========================
# 10. Build OpenAI Batch API input
# =========================

filtered_rows = read_jsonl(FILTERED_SNIPPETS_PATH)

# Keep some margin because final rejection happens after generation
batch_candidates = filtered_rows[: min(len(filtered_rows), int(TARGET_N * 1.5))]

with open(BATCH_INPUT_PATH, "w", encoding="utf-8") as f:
    for row in batch_candidates:
        req = {
            "custom_id": row["id"],
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": OPENAI_MODEL,
                "temperature": OPENAI_TEMPERATURE,
                "max_tokens": OPENAI_MAX_OUTPUT_TOKENS,
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": make_user_prompt(row["code"])},
                ],
            },
        }
        f.write(json.dumps(req, ensure_ascii=False) + "\n")

print(f"Batch input written to: {BATCH_INPUT_PATH}")
print(f"Batch candidates: {len(batch_candidates):,}")

In [ ]:
# =========================
# 11. Upload batch input file
# =========================

batch_input_file = client.files.create(
    file=open(BATCH_INPUT_PATH, "rb"),
    purpose="batch"
)

print("Batch input file id:", batch_input_file.id)

In [ ]:
# =========================
# 12. Create batch job
# =========================

batch_job = client.batches.create(
    input_file_id=batch_input_file.id,
    endpoint="/v1/chat/completions",
    completion_window="24h",
    metadata={"job_name": "oss_instruct_instruction_generation"}
)

print("Batch job id:", batch_job.id)
print("Status:", batch_job.status)

In [ ]:
# =========================
# 13. Check batch status
# =========================

# Replace with your actual batch id if resuming later:
BATCH_ID = batch_job.id

batch_status = client.batches.retrieve(BATCH_ID)
batch_status

In [ ]:
# =========================
# 14. Download completed batch output
# =========================

# Run this after the batch job status becomes 'completed'
batch_status = client.batches.retrieve(BATCH_ID)

print("Current status:", batch_status.status)

if batch_status.status != "completed":
    print("Batch is not completed yet.")
else:
    output_file_id = batch_status.output_file_id
    content = client.files.content(output_file_id).text

    with open(BATCH_OUTPUT_RAW_PATH, "w", encoding="utf-8") as f:
        f.write(content)

    print(f"Saved raw batch output to: {BATCH_OUTPUT_RAW_PATH}")

In [ ]:
# =========================
# 15. Parse batch results
# =========================

source_rows = {row["id"]: row for row in batch_candidates}

parsed = []
parse_rejects = []

def clean_instruction(text: str) -> str:
    text = text.strip()
    text = re.sub(r"^['\"]|['\"]$", "", text)
    text = re.sub(r"^instruction\s*:\s*", "", text, flags=re.IGNORECASE)
    text = text.strip()
    return text

raw_lines = read_jsonl(BATCH_OUTPUT_RAW_PATH)

for item in tqdm(raw_lines):
    custom_id = item.get("custom_id")

    try:
        content = item["response"]["body"]["choices"][0]["message"]["content"]
    except Exception:
        parse_rejects.append({"id": custom_id, "stage": "parse_batch", "reason": "missing_generation"})
        continue

    instruction = clean_instruction(content)

    if not instruction:
        parse_rejects.append({"id": custom_id, "stage": "parse_batch", "reason": "empty_instruction"})
        continue

    if len(instruction) < 15:
        parse_rejects.append({"id": custom_id, "stage": "parse_batch", "reason": "too_short_instruction"})
        continue

    if len(instruction) > 500:
        parse_rejects.append({"id": custom_id, "stage": "parse_batch", "reason": "too_long_instruction"})
        continue

    if any(bad in instruction.lower() for bad in ["the code above", "the snippet above", "this code", "above code"]):
        parse_rejects.append({"id": custom_id, "stage": "parse_batch", "reason": "meta_reference"})
        continue

    if "```" in instruction:
        parse_rejects.append({"id": custom_id, "stage": "parse_batch", "reason": "markdown_fence"})
        continue

    source = source_rows.get(custom_id)
    if source is None:
        parse_rejects.append({"id": custom_id, "stage": "parse_batch", "reason": "missing_source_row"})
        continue

    parsed.append({
        "id": custom_id,
        "instruction": instruction,
        "response": source["code"],
    })

write_jsonl(BATCH_PARSED_PATH, parsed)
append_jsonl(REJECTS_PATH, parse_rejects)

print(f"Parsed usable pairs: {len(parsed):,}")
print(f"Saved to: {BATCH_PARSED_PATH}")

In [ ]:
# =========================
# 16. Final cleanup and deduplication
# =========================

parsed_rows = read_jsonl(BATCH_PARSED_PATH)

final_rows = []
final_rejects = []

seen_pair_hashes = set()
seen_instruction_hashes = set()

for row in tqdm(parsed_rows):
    instruction = normalize_whitespace(row["instruction"])
    response = normalize_whitespace(row["response"])

    if not instruction.endswith((".", "?", ":")):
        # Not required, just makes prompts look more natural
        instruction = instruction + "."

    if not parseable_python(response):
        final_rejects.append({"id": row["id"], "stage": "final_cleanup", "reason": "response_not_parseable"})
        continue

    pair_hash = sha256_text(instruction + "\n---\n" + response)
    instr_hash = sha256_text(instruction)

    if pair_hash in seen_pair_hashes:
        final_rejects.append({"id": row["id"], "stage": "final_cleanup", "reason": "duplicate_pair"})
        continue

    # Optional: keep instruction diversity
    if instr_hash in seen_instruction_hashes:
        final_rejects.append({"id": row["id"], "stage": "final_cleanup", "reason": "duplicate_instruction"})
        continue

    seen_pair_hashes.add(pair_hash)
    seen_instruction_hashes.add(instr_hash)

    final_rows.append({
        "instruction": instruction,
        "response": response
    })

append_jsonl(REJECTS_PATH, final_rejects)

# Truncate to target size
final_rows = final_rows[:TARGET_N]

write_jsonl(FINAL_DATASET_PATH, final_rows)

print(f"Final dataset size: {len(final_rows):,}")
print(f"Saved to: {FINAL_DATASET_PATH}")

In [ ]:
# =========================
# 17. Quick inspection
# =========================

final_rows = read_jsonl(FINAL_DATASET_PATH)

print("Example count:", len(final_rows))
print()

for i in range(min(3, len(final_rows))):
    print("=" * 80)
    print("INSTRUCTION:")
    print(final_rows[i]["instruction"])
    print()
    print("RESPONSE:")
    print(final_rows[i]["response"][:800])
    print()

In [ ]:
# =========================
# 18. Summary stats
# =========================

final_rows = read_jsonl(FINAL_DATASET_PATH)

df = pd.DataFrame(final_rows)
df["instruction_chars"] = df["instruction"].str.len()
df["response_chars"] = df["response"].str.len()
df["response_lines"] = df["response"].str.count("\n") + 1

df[["instruction_chars", "response_chars", "response_lines"]].describe()

## Final output format

The final file contains only:

```json
{"instruction": "...", "response": "..."}
```

This keeps the schema aligned with your request and makes it easy to convert into whatever fine-tuning format you use later.

In [ ]:
# =========================
# 19. Optional Hugging Face upload prep
# =========================

# This cell does NOT upload automatically.
# It just prepares a Hugging Face Dataset object if you want to push later.

from datasets import Dataset

final_rows = read_jsonl(FINAL_DATASET_PATH)
hf_ds = Dataset.from_list(final_rows)
hf_ds

## Important caveats for your thesis

Because this OSS-Instruct pipeline starts from **real open-source code**, it may differ from Self-Instruct not only in task-generation style, but also in the realism and diversity of the code substrate.

That is acceptable for your design as long as you clearly document:

- the source dataset,
- your filtering rules,
- your final schema,
- your target size,
- and your downstream fine-tuning/evaluation controls.